In [1]:
# =========================================================
# FINAL PIPELINE (ConvNeXt-Large + BOOSTED INFERENCE)
# =========================================================

import os, random, glob, time, copy
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets, models

from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
from torch.amp import autocast, GradScaler

# ================= CONFIG =================
SEED = 42
BATCH_SIZE = 8
ACCUM_STEPS = 4
N_FOLDS = 3

HEAD_LR = 3e-4
BLOCK4_LR = 1e-4
BLOCK3_LR = 5e-5

WEIGHT_DECAY = 0.05

TIME_LIMIT = 5 * 3600
START_TIME = time.time()

DEVICE = torch.device("cuda")

ROOT = "/kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors"
TRAIN_DIR = os.path.join(ROOT, "train")
TEST_DIR = os.path.join(ROOT, "test")

# ================= SEED =================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)

# ================= AUG =================
def get_tfms(sz, train=True, stage=1):
    mean = [0.485,0.456,0.406]
    std = [0.229,0.224,0.225]

    if train:
        if stage == 1:
            return transforms.Compose([
                transforms.RandomResizedCrop(sz, scale=(0.8,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(0.2,0.2,0.2,0.1),
                transforms.ToTensor(),
                transforms.Normalize(mean,std),
            ])
        else:
            return transforms.Compose([
                transforms.RandomResizedCrop(sz, scale=(0.6,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.RandAugment(2,7),
                transforms.ToTensor(),
                transforms.Normalize(mean,std),
            ])
    else:
        return transforms.Compose([
            transforms.Resize(int(sz*1.1)),
            transforms.CenterCrop(sz),
            transforms.ToTensor(),
            transforms.Normalize(mean,std),
        ])

# ================= MIXUP =================
def mixup(x,y,alpha=0.2):
    if np.random.rand() < 0.5:
        lam = np.random.beta(alpha,alpha)
        idx = torch.randperm(x.size(0)).to(x.device)
        return lam*x + (1-lam)*x[idx], y, y[idx], lam
    return x,y,y,1.0

# ================= MODEL =================
class ConvNeXtLargeCustom(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        base = models.convnext_large(weights=models.ConvNeXt_Large_Weights.IMAGENET1K_V1)

        self.block0 = nn.Sequential(base.features[0])
        self.block1 = nn.Sequential(base.features[1])
        self.block2 = nn.Sequential(base.features[2])
        self.block3 = nn.Sequential(base.features[3])
        self.block4 = nn.Sequential(base.features[4:])

        in_features = base.classifier[2].in_features

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),   # FIX
            nn.Flatten(),
            nn.Linear(in_features, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.55),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        x = self.block0(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return self.head(x)

# ================= FREEZE =================
def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_block(block):
    for p in block.parameters():
        p.requires_grad = True

# ================= DATA =================
full_ds = datasets.ImageFolder(TRAIN_DIR)
num_classes = len(full_ds.classes)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
test_paths = sorted(glob.glob(os.path.join(TEST_DIR, "*.*")))

all_probs = []

# ================= TRAIN =================
for fold, (t_idx, v_idx) in enumerate(skf.split(np.zeros(len(full_ds)), full_ds.targets)):

    if time.time() - START_TIME > TIME_LIMIT:
        break

    print(f"\n==== FOLD {fold+1} ====")

    model = ConvNeXtLargeCustom(num_classes).to(DEVICE)
    scaler = GradScaler("cuda")
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # -------- STAGE 1 --------
    freeze_all(model)
    for p in model.head.parameters():
        p.requires_grad = True

    optimizer = torch.optim.AdamW(model.head.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)

    train_loader = DataLoader(
        Subset(datasets.ImageFolder(TRAIN_DIR, get_tfms(224, True, 1)), t_idx),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
    )

    for epoch in range(3):
        model.train()
        optimizer.zero_grad()

        for i,(x,y) in enumerate(tqdm(train_loader, leave=False)):
            x,y = x.to(DEVICE), y.to(DEVICE)
            x,y1,y2,lam = mixup(x,y)

            with autocast("cuda"):
                out = model(x)
                loss = lam*criterion(out,y1)+(1-lam)*criterion(out,y2)

            scaler.scale(loss/ACCUM_STEPS).backward()

            if (i+1)%ACCUM_STEPS==0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    # -------- STAGE 2 --------
    unfreeze_block(model.block3)
    unfreeze_block(model.block4)

    optimizer = torch.optim.AdamW([
        {"params": model.block3.parameters(), "lr": BLOCK3_LR},
        {"params": model.block4.parameters(), "lr": BLOCK4_LR},
        {"params": model.head.parameters(), "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

    train_loader = DataLoader(
        Subset(datasets.ImageFolder(TRAIN_DIR, get_tfms(288, True, 2)), t_idx),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
    )

    val_loader = DataLoader(
        Subset(datasets.ImageFolder(TRAIN_DIR, get_tfms(288, False)), v_idx),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2
    )

    best_acc = 0
    best_model = None

    for epoch in range(5):
        model.train()
        optimizer.zero_grad()

        for i,(x,y) in enumerate(tqdm(train_loader, leave=False)):
            x,y = x.to(DEVICE), y.to(DEVICE)
            x,y1,y2,lam = mixup(x,y)

            with autocast("cuda"):
                out = model(x)
                loss = lam*criterion(out,y1)+(1-lam)*criterion(out,y2)

            scaler.scale(loss/ACCUM_STEPS).backward()

            if (i+1)%ACCUM_STEPS==0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        scheduler.step()

        # VALIDATE
        model.eval()
        correct,total = 0,0

        with torch.no_grad():
            for x,y in val_loader:
                x,y = x.to(DEVICE), y.to(DEVICE)
                with autocast("cuda"):
                    out = model(x)
                correct += (out.argmax(1)==y).sum().item()
                total += y.size(0)

        acc = correct/total
        print(f"Epoch {epoch+1} Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc = acc
            best_model = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model)

    # ================= BOOSTED TTA =================
    model.eval()
    fold_probs = []

    tf1 = get_tfms(288, False)
    tf2 = transforms.Compose([
        transforms.Resize(320),
        transforms.CenterCrop(288),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    TEMP = 1.3
    def sharpen(p):
        return torch.softmax(torch.log(p + 1e-8) / TEMP, dim=1)

    with torch.no_grad():
        for p in tqdm(test_paths, leave=False):
            try:
                img = Image.open(p).convert("RGB")

                x1 = tf1(img).unsqueeze(0).to(DEVICE)
                x2 = torch.flip(x1,[3])
                x3 = tf2(img).unsqueeze(0).to(DEVICE)
                x4 = torch.flip(x3,[3])
                x5 = tf1(img.rotate(5)).unsqueeze(0).to(DEVICE)

                with autocast("cuda"):
                    p1 = torch.softmax(model(x1),1)
                    p2 = torch.softmax(model(x2),1)
                    p3 = torch.softmax(model(x3),1)
                    p4 = torch.softmax(model(x4),1)
                    p5 = torch.softmax(model(x5),1)

                p1,p2,p3,p4,p5 = map(sharpen,[p1,p2,p3,p4,p5])

                pred = (1.2*p1 + 1.1*p2 + 1.0*p3 + 1.0*p4 + 0.9*p5) / 5.2

            except:
                pred = torch.zeros((1, num_classes)).to(DEVICE)

            fold_probs.append(pred.cpu().numpy())

    all_probs.append(np.vstack(fold_probs))

    del model
    torch.cuda.empty_cache()

# ================= SUBMIT =================
final_probs = np.mean(all_probs, axis=0)

# 🔥 final sharpening
final_probs = final_probs ** 1.1
final_probs = final_probs / final_probs.sum(axis=1, keepdims=True)

labels = final_probs.argmax(1)

pd.DataFrame({
    "ImageName":[os.path.basename(p) for p in test_paths],
    "label":labels
}).to_csv("submission.csv", index=False)

print("✅ DONE")


==== FOLD 1 ====
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:04<00:00, 194MB/s]


Epoch 1 Acc: 0.4909


Epoch 2 Acc: 0.5080


Epoch 3 Acc: 0.5087


...

Epoch 3 Acc: 0.4928


Epoch 4 Acc: 0.4976


Epoch 5 Acc: 0.4951


/tmp/ipykernel_23/2365706738.py:310: RuntimeWarning: invalid value encountered in divide
  final_probs = final_probs / final_probs.sum(axis=1, keepdims=True)


✅ DONE
